# Deploy IRCv2 to SageMaker Serverless Endpoint

This notebook deploys the IRCv2 model to a SageMaker serverless endpoint.

## Setup

### Import Dependencies

In [ ]:
%matplotlib inline

import boto3
import time
import json
import base64
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import sagemaker
from io import BytesIO

### Initialize AWS Session

In [ ]:
sess = boto3.Session()
sm = sess.client('sagemaker')
region = sess.region_name
account = boto3.client('sts').get_caller_identity().get('Account')

### Get IAM Role

**Note**: Ensure the IAM role has:
- `AmazonS3FullAccess`
- `AmazonSageMakerFullAccess`

In [ ]:
role = sagemaker.get_execution_role()
print(f"Using role: {role}")

## Build and Push Container

Build the custom container with our inference code and push it to Amazon ECR.

In [ ]:
# download the model artifacts from S3
bucket = "animl-model-zoo"
model_key = "irc/tf-saved-model.tar.gz"
model_local_path = Path("/tmp") / "tf-saved-model.tar.gz"
s3 = boto3.client("s3")
s3.download_file(bucket, model_key, str(model_local_path))
print(f"Downloaded model artifacts to: {model_local_path}")

In [ ]:
# unzip the model artifacts and move them to the correct location
import tarfile
with tarfile.open(model_local_path, mode='r:gz') as archive:
    archive.extractall(path="/tmp/irc_model")
print(f"Extracted model artifacts to: /tmp/irc_model")

!mv /tmp/irc_model/export .
print(f"Moved model artifacts to: ./export")

In [ ]:
# Create ECR repository if it doesn't exist
registry_name = "tfserving-irc-sagemaker"
ecr = boto3.client('ecr')

try:
    ecr.create_repository(repositoryName=registry_name)
except ecr.exceptions.RepositoryAlreadyExistsException:
    pass

# Get auth token and login to ECR
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account}.dkr.ecr.{region}.amazonaws.com

# Build container
!docker build -q -t {registry_name} -f Dockerfile .

# Tag and push to ECR
image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{registry_name}:latest"
!docker tag {registry_name} {image_uri}
!docker push {image_uri}

print(f"Container pushed to: {image_uri}")

## Create SageMaker Model

In [ ]:
model_prefix = "ircv2"

# Check if model already exists
model_already_created = False
for model_def in sm.list_models()['Models']:
    if model_prefix == model_def['ModelName']:
        create_model_response = model_def
        model_already_created = True

# Create model if it doesn't exist
if not model_already_created:
    create_model_response = sm.create_model(
        ModelName=model_prefix,
        ExecutionRoleArn=role,
        PrimaryContainer={
            "Image": image_uri,
            "Environment": {
                "SAGEMAKER_PROGRAM": "serve.py"
            }
        }
    )

print(f"Model ARN: {create_model_response['ModelArn']}")

## Create Endpoint Configurations

In [ ]:
# Create realtime and batch endpoint configuration
# for batch endpoints, use concurrency of 80, for real-time endpoints, use 20
# https://github.com/tnc-ca-geo/animl-api/issues/101

# Disable batch endpoint config creation if not needed
create_batch_endpoint_config = True

realtime_endpoint_config_name = f"{model_prefix}-config-concurrency-20"
realtime_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=realtime_endpoint_config_name,
    ProductionVariants=[
        {
            "ModelName": model_prefix,
            "VariantName": "AllTraffic",
            "ServerlessConfig": {
                "MemorySizeInMB": 6144,  # 6GB memory
                "MaxConcurrency": 20     # Maximum concurrent invocations
            }
        }
    ]
)
print(f"Endpoint Config ARN: {realtime_endpoint_config_response['EndpointConfigArn']}")


if create_batch_endpoint_config:
    batch_endpoint_config_name = f"{model_prefix}-config-concurrency-80"
    batch_endpoint_config_response = sm.create_endpoint_config(
        EndpointConfigName=batch_endpoint_config_name,
        ProductionVariants=[
            {
                "ModelName": model_prefix,
                "VariantName": "AllTraffic",
                "ServerlessConfig": {
                    "MemorySizeInMB": 6144,  # 6GB memory
                    "MaxConcurrency": 80     # Maximum concurrent invocations
                }
            }
        ]
    )
    print(f"Endpoint Config ARN: {batch_endpoint_config_response['EndpointConfigArn']}")

## Create and Deploy Endpoints

In [ ]:
# Create realtime endpoint
realtime_endpoint_name = f"{model_prefix}-concurrency-20"
create_realtime_endpoint_response = sm.create_endpoint(
    EndpointName=realtime_endpoint_name,
    EndpointConfigName=realtime_endpoint_config_name
)

print(f"Endpoint ARN: {create_realtime_endpoint_response['EndpointArn']}")

# Wait for endpoint creation
resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
realtime_status = resp['EndpointStatus']
print(f"Status: {realtime_status}")

while realtime_status == 'Creating':
    time.sleep(60)
    resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
    realtime_status = resp['EndpointStatus']
    print(f"Status: {realtime_status}")
    if realtime_status == 'Failed':
        realtime_failure_reason = resp.get('FailureReason', 'No failure reason provided')
        print(f"Realtime endpoint deployment failed: {realtime_failure_reason}")
        break

# Get CloudWatch logs for the endpoint
logs = boto3.client('logs')

print(f"Realtime Arn: {resp['EndpointArn']}")
print(f"Realtime endpoint final status: {realtime_status}")
if realtime_status == 'Failed':
    realtime_log_group = f"/aws/sagemaker/Endpoints/{realtime_endpoint_name}"
    try:
        log_streams = logs.describe_log_streams(logGroupName=realtime_log_group)
        for stream in log_streams['logStreams']:
            print(f"\nLog stream: {stream['logStreamName']}")
            realtime_events = logs.get_log_events(logGroupName=realtime_log_group, logStreamName=stream['logStreamName'])
            for event in realtime_events['events']:
                print(event['message'])
    except Exception as e:
        print(f"Error fetching logs: {str(e)}")

In [ ]:
# Create batch endpoint if needed
batch_endpoint_name = f"{model_prefix}-concurrency-80"
if create_batch_endpoint_config:
    create_batch_endpoint_response = sm.create_endpoint(
        EndpointName=batch_endpoint_name,
        EndpointConfigName=batch_endpoint_config_name
    )

    print(f"Batch Endpoint ARN: {create_batch_endpoint_response['EndpointArn']}")

    # Wait for batch endpoint creation
    batch_resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
    batch_status = batch_resp['EndpointStatus']
    print(f"Status: {batch_status}")

    while batch_status == 'Creating':
        time.sleep(60)
        batch_resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
        batch_status = resp['EndpointStatus']
        print(f"Status: {batch_status}")
        if batch_status == 'Failed':
            failure_reason = resp.get('FailureReason', 'No failure reason provided')
            print(f"Batch endpoint deployment failed: {failure_reason}")
            break
    print(f"Batch Arn: {resp['EndpointArn']}")
    print(f"Batch endpoint final status: {batch_status}")
    if batch_status == 'Failed':
        batch_log_group = f"/aws/sagemaker/Endpoints/{batch_endpoint_config_name}"
        try:
            log_streams = logs.describe_log_streams(logGroupName=batch_log_group)
            for stream in log_streams['logStreams']:
                print(f"\nLog stream: {stream['logStreamName']}")
                batch_events = logs.get_log_events(logGroupName=batch_log_group, logStreamName=stream['logStreamName'])
                for event in batch_events['events']:
                    print(event['message'])
        except Exception as e:
            print(f"Error fetching logs: {str(e)}")

### Util for Visualizing Results

In [ ]:
def draw_detections(image, predictions):
    """Draw bounding boxes and labels on the image.
    """
    # Create a copy of the image to draw on
    image_draw = image.copy()
    draw = ImageDraw.Draw(image_draw)

    # Get image dimensions
    width, height = image.size

    for pred in predictions['predictions']:
        # Draw detections if present
        if 'detections' in pred:
            for det in pred['detections']:
                # Get bounding box coordinates
                bbox = det['bbox']

                x1 = bbox[0] * width
                y1 = bbox[1] * height
                x2 = (bbox[0] + bbox[2]) * width
                y2 = (bbox[1] + bbox[3]) * height

                # Draw bounding box
                draw.rectangle([x1, y1, x2, y2], outline='red', width=3)

                # Add label with confidence
                label = f"{det['category']}: {det['conf']:.2f}"
                draw.text((x1, y1-15), label, fill='red')

    return image_draw

## Test the Endpoint

In [ ]:
# Load a test image from the test-data directory
test_image = Image.open("tests/test-data/coyote-test.jpg")
display(test_image)

# Convert image to base64
buffered = BytesIO()
test_image.save(buffered, format="JPEG")
img_str = base64.b64encode(buffered.getvalue()).decode()

# Prepare payload
payload = {
    "image": img_str,
    "bbox": [0.4319087266921997, 0.21275195479393005, 0.6099987030029297, 0.3272196650505066]
}

# Invoke endpoint
client = boto3.client('runtime.sagemaker')
response = client.invoke_endpoint(
    EndpointName=realtime_endpoint_name,
    ContentType='application/json',
    Body=json.dumps(payload)
)

# Parse results
result = json.loads(response['Body'].read().decode())
print("\nPrediction Results:")
print(json.dumps(result, indent=2))

## Visualize Results

In [ ]:
# Draw detections and classifications on the image
annotated_image = draw_detections(test_image, result)
display(annotated_image)

## Cleanup Resources

**Note**: Only run this cell when you want to delete the endpoint and associated resources.

In [ ]:
# Uncomment to cleanup
# client = boto3.client('sagemaker')
# client.delete_endpoint(EndpointName=realtime_endpoint_name)
# client.delete_endpoint_config(EndpointConfigName=realtime_endpoint_config_name)
# client.delete_model(ModelName=model_prefix)